# SRE Copilot - Complete Code Analysis and Demo Flow

This notebook provides a comprehensive analysis of the SRE Copilot system, including all code components, their locations, and how they work together for the end-to-end demo.

## Table of Contents
1. **System Architecture Overview**
2. **Incident Creation Flow**
3. **EventBridge Rules and Triggers**
4. **Lambda Functions and Bedrock Agents**
5. **Knowledge Base Implementation**
6. **Complete Demo Walkthrough**

## 1. System Architecture Overview

The SRE Copilot is an AI-powered Site Reliability Engineering assistant built on AWS services.

### Key Components:
- **Streamlit Dashboard**: Web interface for incident management
- **AWS Systems Manager**: OpsItems for incident tracking
- **AWS Lambda**: Serverless functions for all processing
- **AWS Bedrock**: AI analysis using Claude 3 Sonnet
- **DynamoDB**: Serverless knowledge base with vector search
- **CloudWatch**: Metrics and logs monitoring
- **EventBridge**: Event-driven automation

## 2. Incident Creation Flow

### Important Note: No Grafana Integration or Automatic CloudWatch Alarms

The system does NOT integrate with Grafana or automatically create CloudWatch alarms. Instead, it uses a manual incident creation process.

### 2.1 Streamlit Dashboard - Main Entry Point

**Location**: `/home/ec2-user/sre/sre_mcp/streamlit_app.py`

The main dashboard application that provides the user interface for incident management.

In [ ]:
# Key code from streamlit_app.py for incident creation

class SRECopilotDashboard:
    def __init__(self):
        """Initialize AWS clients and configuration."""
        self.clients = {
            'ssm': boto3.client('ssm'),
            'cloudwatch': boto3.client('cloudwatch'),
            'logs': boto3.client('logs'),
            'lambda': boto3.client('lambda')
        }
    
    def create_opsitem(self, title, description, severity='3'):
        """Create SSM OpsItem."""
        try:
            response = self.clients['ssm'].create_ops_item(
                Title=title,
                Description=description,
                Priority=int(severity),
                Source='SRE-Demo-Streamlit',
                Category='Performance',
                Severity=severity,
                OperationalData={
                    'start_time': {'Value': datetime.utcnow().isoformat()},
                    'service': {'Value': 'sre-demo-app'},
                    'environment': {'Value': 'demo'}
                }
            )
            return response['OpsItemId']
        except Exception as e:
            st.error(f"Error creating OpsItem: {str(e)}")
            return None

### 2.2 Incident Generation Flow

When a user clicks "Generate Random Incident" in the dashboard:

In [ ]:
# Code from streamlit_app.py - Line 1210

def generate_scenario_incident(self, scenario):
    """Generate incident based on scenario."""
    # Create CloudWatch metrics
    namespace = 'SREDemo/Application'
    
    # Generate CPU spike for performance issues
    if scenario['type'] == 'performance':
        self.clients['cloudwatch'].put_metric_data(
            Namespace=namespace,
            MetricData=[
                {
                    'MetricName': 'CPUUtilization',
                    'Value': 95.0,
                    'Unit': 'Percent',
                    'Timestamp': datetime.utcnow()
                }
            ]
        )
    
    # Create CloudWatch logs with errors
    log_group = '/aws/lambda/sre-demo-app'
    log_stream = f"demo-stream-{datetime.utcnow().strftime('%Y%m%d%H%M%S')}"
    
    self.clients['logs'].create_log_stream(
        logGroupName=log_group,
        logStreamName=log_stream
    )
    
    # Put error logs
    error_messages = [
        "ERROR: Connection timeout to database",
        "ERROR: High memory utilization detected",
        "CRITICAL: Service degradation in progress"
    ]
    
    # Create OpsItem
    title = f"{scenario['name']} - {datetime.utcnow().strftime('%Y-%m-%d %H:%M')}"
    description = scenario['description']
    ops_item_id = self.create_opsitem(title, description, scenario['severity'])
    
    return {
        'ops_item_id': ops_item_id,
        'title': title,
        'type': scenario['type'],
        'severity': scenario['severity']
    }

## 3. EventBridge Rules and Triggers

### 3.1 EventBridge Rule for Auto-Indexing

**Location**: `/home/ec2-user/sre/sre_mcp/deploy_knowledge_base_serverless.sh`

In [ ]:
# EventBridge rule creation from deploy_knowledge_base_serverless.sh - Line 151

# Create CloudWatch Events rule for OpsItem auto-indexing
aws events put-rule \
    --name sre-opsitem-indexing \
    --description "Auto-index OpsItems to knowledge base" \
    --event-pattern '{
        "source": ["aws.ssm"],
        "detail-type": ["AWS API Call via CloudTrail"],
        "detail": {
            "eventName": ["CreateOpsItem", "UpdateOpsItem"]
        }
    }' \
    --region us-east-1

# Add Lambda permission
aws lambda add-permission \
    --function-name sre-knowledge-base-agent-lambda \
    --statement-id AllowEventsInvoke \
    --action lambda:InvokeFunction \
    --principal events.amazonaws.com \
    --source-arn arn:aws:events:us-east-1:*:rule/sre-opsitem-indexing

# Add Lambda as target
aws events put-targets \
    --rule sre-opsitem-indexing \
    --targets "Id"="1","Arn"="arn:aws:lambda:us-east-1:*:function:sre-knowledge-base-agent-lambda"

### 3.2 Understanding the Enhanced EventBridge Flow for Automatic AI Analysis

**MAJOR UPDATE**: The EventBridge rule now triggers AUTOMATIC AI-powered root cause analysis!

Here's the enhanced flow:

1. **EventBridge Rule** (`sre-opsitem-indexing`) monitors CloudTrail for OpsItem creation/update
2. **Triggers Enhanced OpsItem Indexer** (`sre-opsitem-indexer`) Lambda
3. **Enhanced OpsItem Indexer** performs THREE actions:
   - Indexes OpsItem to Knowledge Base for future searches
   - **NEW**: Automatically invokes Supervisor Lambda for AI analysis
   - **NEW**: Updates OpsItem with analysis results
4. **Supervisor Lambda** orchestrates comprehensive analysis:
   - Collects CloudWatch metrics and logs
   - Searches Knowledge Base for similar incidents
   - **NEW**: Automatically invokes ALL specialized agent Lambdas
   - Generates AI-powered root cause analysis using Bedrock
5. **Results** are automatically stored:
   - Analysis added to OpsItem operational data
   - AI analysis indexed to Knowledge Base
   - Complete audit trail maintained

**No manual intervention required!** The entire AI analysis happens automatically when an incident is created.

### 3.3 OpsItem Indexer Lambda - The EventBridge Handler

**Location**: `/home/ec2-user/sre/sre_mcp/src/lambdas/opsitem-indexer/lambda_function.py`

This is the Lambda function that EventBridge actually triggers. It acts as an intermediary between EventBridge and the Knowledge Base Lambda.

In [ ]:
# Enhanced OpsItem Indexer Lambda - Now with Automatic AI Analysis!
# From /src/lambdas/opsitem-indexer/lambda_function_enhanced.py

def lambda_handler(event, context):
    """
    Enhanced Lambda that automatically triggers full AI analysis.
    """
    # Extract OpsItem ID from CloudTrail event
    ops_item_id = extract_ops_item_id(event)
    
    # Get full OpsItem details
    ops_item = ssm_client.get_ops_item(OpsItemId=ops_item_id)['OpsItem']
    
    # Step 1: Index to knowledge base
    kb_result = index_to_knowledge_base(ops_item)
    
    # Step 2: NEW - Trigger automatic AI analysis for new incidents
    if ops_item['Status'] not in ['Resolved', 'Closed']:
        # Collect initial data
        collected_data = collect_initial_data(ops_item)
        
        # Invoke supervisor Lambda for full AI analysis
        analysis_result = invoke_supervisor_analysis(ops_item, collected_data)
        
        # Update OpsItem with AI analysis results
        if analysis_result:
            update_opsitem_with_analysis(ops_item_id, analysis_result)
            store_analysis_in_kb(ops_item_id, ops_item, analysis_result)
            
    return {
        'statusCode': 200,
        'body': json.dumps({
            'indexed': True,
            'analyzed': True,
            'message': 'AI analysis completed automatically'
        })
    }

def invoke_supervisor_analysis(ops_item, collected_data):
    """Automatically trigger supervisor Lambda for AI analysis."""
    payload = {
        'action': 'analyze',
        'incident_description': f"{ops_item['Title']}. {ops_item['Description']}",
        'service': ops_item.get('OperationalData', {}).get('service', {}).get('Value', 'sre-demo-app'),
        'enable_kb': True,
        'additional_context': {
            'ops_item_id': ops_item['OpsItemId'],
            'source': 'EventBridge-Auto-Analysis'  # Identifies automatic trigger
        }
    }
    
    # Invoke supervisor for comprehensive analysis
    response = lambda_client.invoke(
        FunctionName='sre-supervisor-lambda',
        InvocationType='RequestResponse',
        Payload=json.dumps(payload)
    )
    
    return parse_analysis_results(response)

### 3.4 Knowledge Base Lambda - Processing Index Requests

The Knowledge Base Lambda receives requests from the OpsItem Indexer and performs the actual indexing:

In [ ]:
# Knowledge Base Lambda Handler
# From /src/lambdas/knowledge-base-agent/lambda_function_serverless.py

def lambda_handler(event, context):
    """Lambda handler for serverless knowledge base operations."""
    kb = ServerlessKnowledgeBase()
    action = event.get('action', 'search')
    
    if action == 'index_opsitem':
        # Called by OpsItem Indexer Lambda
        ops_item = event.get('ops_item')
        result = kb.index_opsitem(ops_item)
        return {
            'statusCode': 200,
            'body': json.dumps(result)
        }
    
    elif action == 'search':
        # Called by Supervisor Lambda for similarity search
        query = event.get('query', '')
        category = event.get('category')
        k = event.get('k', 5)
        
        results = kb.search_similar_documents(query, category, k)
        return {
            'statusCode': 200,
            'body': json.dumps({
                'results': results,
                'count': len(results)
            })
        }

def index_opsitem(self, ops_item: Dict[str, Any]):
    """Index an OpsItem to the knowledge base."""
    # Create document from OpsItem
    document = {
        'document_id': f"opsitem-{ops_item.get('OpsItemId')}",
        'title': ops_item.get('Title', ''),
        'content': f"{ops_item.get('Title', '')}\\n\\n{ops_item.get('Description', '')}",
        'metadata': {
            'category': 'incident',
            'type': 'opsitem',
            'severity': ops_item.get('Severity', '3'),
            'status': ops_item.get('Status', 'Open'),
            'tags': ['auto-indexed', 'incident'],
            'source': 'systems-manager',
            'created_time': ops_item.get('CreatedTime', ''),
            'ops_item_id': ops_item.get('OpsItemId')
        }
    }
    
    # If resolved, add resolution details
    if ops_item.get('Status') in ['Resolved', 'Closed']:
        operational_data = ops_item.get('OperationalData', {})
        if '/aws/resolution' in operational_data:
            document['metadata']['resolution'] = operational_data['/aws/resolution'].get('Value')
    
    # Index the document with embeddings
    return self.index_document(document)

In [ ]:
# Code from streamlit_app.py - Line 945

def invoke_supervisor_analysis(self, ops_item, collected_data):
    """Invoke the supervisor agent for analysis with optional MCP."""
    
    # Build the payload for supervisor Lambda
    payload = {
        'action': 'analyze',
        'incident_description': f"{ops_item.get('Title', '')}. {ops_item.get('Description', '')}",
        'start_time': (datetime.utcnow() - timedelta(hours=1)).isoformat(),
        'end_time': datetime.utcnow().isoformat(),
        'service': ops_item.get('OperationalData', {}).get('service', {}).get('Value', 'sre-demo-app'),
        'environment': 'demo',
        'enable_mcp': st.session_state.get('mcp_enabled', False),
        'enable_kb': True,  # Enable knowledge base search
        'additional_context': {
            'ops_item_id': ops_item.get('OpsItemId'),
            'severity': ops_item.get('Severity'),
            'data_summary': {
                'log_events': sum(len(events) for events in collected_data['logs'].values()),
                'metric_points': sum(len(points) for points in collected_data['metrics'].values())
            }
        }
    }
    
    # Invoke the supervisor Lambda
    response = self.lambda_client.invoke(
        FunctionName='sre-supervisor-lambda',
        InvocationType='RequestResponse',
        Payload=json.dumps(payload)
    )
    
    # Parse response
    result = json.loads(response['Payload'].read())
    return result

## 4. Lambda Functions and Bedrock Agents

### 4.1 Supervisor Lambda - Main Orchestrator

**Location**: `/home/ec2-user/sre/sre_mcp/src/lambdas/supervisor/lambda_function.py`

### 4.0 Enhanced Supervisor Lambda - Automatic Agent Orchestration

The supervisor Lambda has been enhanced to automatically invoke ALL specialized agents:

In [ ]:
# Enhanced Supervisor Lambda - Automatically invokes all agents
# From /src/lambdas/supervisor/lambda_function_auto_analysis.py

# List of ALL specialized agents
SPECIALIZED_AGENTS = [
    {
        'name': 'cloudwatch-logs-agent',
        'applicable_for': ['performance', 'outage', 'general'],
        'action': 'analyze_log_group'
    },
    {
        'name': 'cloudtrail-agent',
        'applicable_for': ['security', 'general'],
        'action': 'get_security_events'
    },
    {
        'name': 'vpc-agent',
        'applicable_for': ['security', 'network', 'outage'],
        'action': 'analyze_vpc_issues'
    },
    {
        'name': 'vpc-flow-logs-agent',
        'applicable_for': ['security', 'network', 'performance'],
        'action': 'analyze_flow_logs'
    },
    {
        'name': 'trusted-advisor-agent',
        'applicable_for': ['performance', 'security', 'cost', 'general'],
        'action': 'get_recommendations'
    },
    {
        'name': 'personal-health-agent',
        'applicable_for': ['outage', 'performance', 'general'],
        'action': 'get_health_events'
    }
]

def lambda_handler(event, context):
    """Enhanced handler that automatically orchestrates all agents."""
    
    # Determine incident type
    incident_type = analyze_incident_type(incident_description)
    
    # Step 1: Collect CloudWatch metrics
    metrics_data = get_demo_metrics()
    
    # Step 2: Collect CloudWatch logs
    log_data = get_demo_logs()
    
    # Step 3: Search Knowledge Base
    kb_context = get_knowledge_base_context(incident_description, incident_type)
    
    # Step 4: NEW - Automatically invoke ALL applicable agents!
    agent_analyses = invoke_all_specialized_agents(
        incident_description, 
        incident_type, 
        service
    )
    
    # Step 5: Generate comprehensive AI analysis with ALL data
    context_data = {
        'metrics_data': metrics_data,
        'log_data': log_data,
        'kb_context': kb_context,
        'agent_analyses': agent_analyses  # All agent findings included!
    }
    
    ai_analysis = analyze_with_bedrock(incident_description, context_data, incident_type)
    
    # Step 6: Generate comprehensive report
    comprehensive_analysis = generate_comprehensive_report(
        incident_description, incident_type, ai_analysis,
        agent_analyses, metrics_data, log_data, kb_context
    )
    
    return {
        'statusCode': 200,
        'body': json.dumps({
            'ai_analysis': ai_analysis,
            'comprehensive_analysis': comprehensive_analysis,
            'agent_analyses': agent_analyses,
            'analysis_summary': {
                'agents_invoked': len(agent_analyses),
                'agents_responded': sum(1 for a in agent_analyses.values() if a['success']),
                'source': event.get('additional_context', {}).get('source', 'manual')
            }
        })
    }

In [ ]:
# Supervisor Lambda handler

def lambda_handler(event, context):
    """Enhanced Lambda handler for supervisor agent with AI-powered analysis."""
    try:
        logger.info(f"Received event: {json.dumps(event)}")
        
        # Extract incident information
        action = event.get('action', '')
        incident_description = event.get('incident_description', '')
        service = event.get('service', 'unknown')
        environment = event.get('environment', 'unknown')
        additional_context = event.get('additional_context', {})
        enable_kb = event.get('enable_kb', True)
        mask_ips = event.get('mask_ips', True)
        
        # Determine incident type
        incident_type = analyze_incident_type(incident_description)
        logger.info(f"Detected incident type: {incident_type}")
        
        # Get actual metrics from CloudWatch
        logger.info("Gathering demo metrics...")
        metrics_data = get_demo_metrics()
        
        # Get actual logs from CloudWatch
        logger.info(f"Gathering demo logs (IP masking: {mask_ips})...")
        log_data = get_demo_logs(mask_ips=mask_ips)
        
        # Get knowledge base context if enabled
        kb_context = {}
        if enable_kb:
            logger.info("Querying knowledge base for context...")
            kb_context = get_knowledge_base_context(incident_description, incident_type)
        
        # Prepare context for AI analysis
        context_data = {
            'metrics_data': metrics_data,
            'log_data': log_data,
            'kb_context': kb_context
        }
        
        # Get AI-powered analysis from Bedrock
        logger.info("Generating AI-powered root cause analysis...")
        ai_analysis = analyze_with_bedrock(incident_description, context_data, incident_type)
        
        # Invoke specialized agents based on incident type
        agent_analyses = {}
        if incident_type == 'security':
            agent_analyses['cloudtrail'] = invoke_cloudtrail_agent(incident_description)
            agent_analyses['vpc'] = invoke_vpc_agent(incident_description)
        elif incident_type == 'performance':
            agent_analyses['cloudwatch_logs'] = invoke_cloudwatch_logs_agent(incident_description)
        
        # Build comprehensive response
        response_body = {
            'incident_type': incident_type,
            'ai_analysis': ai_analysis,
            'agent_analyses': agent_analyses,
            'metrics_data': metrics_data,
            'log_data': log_data,
            'kb_context': kb_context,
            'timestamp': datetime.utcnow().isoformat()
        }
        
        return {
            'statusCode': 200,
            'body': json.dumps(response_body)
        }
        
    except Exception as e:
        logger.error(f"Error in lambda_handler: {str(e)}")
        return {
            'statusCode': 500,
            'body': json.dumps({'error': str(e)})
        }

### 4.2 AI Analysis with Bedrock

The supervisor uses AWS Bedrock with Claude 3 Sonnet for intelligent analysis:

In [ ]:
def analyze_with_bedrock(incident_description, context_data, incident_type=None):
    """Analyze incident using AWS Bedrock with Claude 3 Sonnet."""
    try:
        # Prepare context summary
        context_summary = {
            "incident_type": incident_type,
            "metrics": {},
            "logs": {},
            "knowledge_base": {}
        }
        
        # Extract key metrics
        if 'metrics_data' in context_data:
            metrics = context_data['metrics_data']
            for metric_name, data in metrics.items():
                if data and 'latest' in data:
                    latest = data['latest']
                    context_summary['metrics'][metric_name] = {
                        'average': latest.get('Average', 0),
                        'maximum': latest.get('Maximum', 0),
                        'minimum': latest.get('Minimum', 0)
                    }
        
        # Extract log summary
        if 'log_data' in context_data:
            logs = context_data['log_data']
            context_summary['logs'] = {
                'error_count': logs.get('error_count', 0),
                'warning_count': logs.get('warning_count', 0),
                'recent_errors': logs.get('error_messages', [])[:5]
            }
        
        # Include KB context if available
        if 'kb_context' in context_data and context_data['kb_context'].get('similar_incidents'):
            context_summary['knowledge_base'] = {
                'similar_incidents_count': len(context_data['kb_context']['similar_incidents']),
                'top_resolution': context_data['kb_context']['similar_incidents'][0].get('resolution', 'N/A')
            }
        
        # Prepare the prompt for Claude
        prompt = f"""You are an expert SRE analyzing a production incident. 

Incident Description: {incident_description}
Incident Type: {incident_type or 'Unknown'}

Context Data:
{json.dumps(context_summary, indent=2)}

Based on the metrics, logs, and knowledge base context, please provide:

1. **Root Cause Analysis**: Identify the most likely root cause based on the evidence
2. **Impact Assessment**: Describe the business and technical impact
3. **Immediate Mitigation Steps**: List 3-5 actionable steps to resolve the issue NOW
4. **Long-term Recommendations**: Suggest improvements to prevent recurrence
5. **Similar Incidents**: Note any patterns from past incidents if available

Be specific, technical, and actionable. Focus on the evidence from metrics and logs."""

        # Call Bedrock with Claude 3 Sonnet
        response = bedrock_runtime.invoke_model(
            modelId='anthropic.claude-3-sonnet-20240229-v1:0',
            contentType='application/json',
            accept='application/json',
            body=json.dumps({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 2000,
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                "temperature": 0.3  # Lower temperature for consistent analysis
            })
        )
        
        result = json.loads(response['body'].read())
        ai_analysis = result.get('content', [{}])[0].get('text', '')
        
        return ai_analysis
            
    except Exception as e:
        logger.error(f"Error in analyze_with_bedrock: {str(e)}")
        return None

### 4.3 Specialized Agent Lambda Functions

The system includes 7 specialized agent Lambda functions:

In [ ]:
# List of all Lambda functions and their purposes

LAMBDA_FUNCTIONS = [
    {
        "name": "sre-supervisor-lambda",
        "location": "/src/lambdas/supervisor/lambda_function.py",
        "purpose": "Main orchestrator for root cause analysis",
        "triggers": "Manual invocation from Streamlit dashboard"
    },
    {
        "name": "sre-opsitem-indexer",
        "location": "/src/lambdas/opsitem-indexer/lambda_function.py",
        "purpose": "Automatically indexes OpsItems to knowledge base",
        "triggers": "EventBridge rule (sre-opsitem-indexing) on CreateOpsItem/UpdateOpsItem"
    },
    {
        "name": "sre-knowledge-base-agent-lambda",
        "location": "/src/lambdas/knowledge-base-agent/lambda_function_serverless.py",
        "purpose": "Manages knowledge base - indexes documents and performs vector searches",
        "triggers": "Called by opsitem-indexer for indexing, supervisor for searching"
    },
    {
        "name": "sre-cloudwatch-logs-agent-lambda",
        "location": "/src/lambdas/cloudwatch_logs_agent/lambda_function.py",
        "purpose": "Analyzes CloudWatch logs for errors and patterns",
        "triggers": "Invoked by supervisor Lambda"
    },
    {
        "name": "sre-cloudtrail-agent-lambda",
        "location": "/src/lambdas/cloudtrail_agent/lambda_function.py",
        "purpose": "Analyzes CloudTrail events for security issues",
        "triggers": "Invoked by supervisor Lambda"
    },
    {
        "name": "sre-vpc-agent-lambda",
        "location": "/src/lambdas/vpc_agent/lambda_function.py",
        "purpose": "Analyzes VPC configurations and security groups",
        "triggers": "Invoked by supervisor Lambda"
    },
    {
        "name": "sre-vpc-flow-logs-agent-lambda",
        "location": "/src/lambdas/vpc_flow_logs_agent/lambda_function.py",
        "purpose": "Analyzes VPC Flow Logs for network issues",
        "triggers": "Invoked by supervisor Lambda"
    },
    {
        "name": "sre-trusted-advisor-agent-lambda",
        "location": "/src/lambdas/trusted_advisor_agent/lambda_function.py",
        "purpose": "Gets Trusted Advisor recommendations",
        "triggers": "Invoked by supervisor Lambda"
    },
    {
        "name": "sre-personal-health-agent-lambda",
        "location": "/src/lambdas/personal_health_agent/lambda_function.py",
        "purpose": "Checks AWS Personal Health Dashboard",
        "triggers": "Invoked by supervisor Lambda"
    }
]

### 4.4 Bedrock Agent Configuration

**Location**: `/home/ec2-user/sre/sre_mcp/configure_bedrock_agents.py`

In [ ]:
# Bedrock agent configuration

BEDROCK_AGENTS = [
    {
        "agent_name": "SRE-Supervisor",
        "agent_resource_role_arn": f"arn:aws:iam::{account_id}:role/AmazonBedrockExecutionRoleForAgents_{account_id}",
        "customer_encryption_key_arn": "",
        "description": "Main SRE supervisor agent for incident analysis",
        "foundation_model": "anthropic.claude-v2",
        "idle_session_ttl_in_seconds": 3600,
        "instruction": """You are the main SRE supervisor agent responsible for coordinating 
                          incident analysis. When given an incident, you should:
                          1. Analyze the incident description
                          2. Determine which specialized agents to invoke
                          3. Coordinate the analysis from multiple agents
                          4. Provide a comprehensive root cause analysis""",
        "action_groups": [
            {
                "action_group_name": "supervisor-actions",
                "action_group_executor": {
                    "lambda": f"arn:aws:lambda:{region}:{account_id}:function:sre-supervisor-lambda"
                },
                "api_schema": {
                    "openapi": "3.0.0",
                    "info": {
                        "title": "Supervisor Actions API",
                        "version": "1.0.0"
                    },
                    "paths": {
                        "/analyze": {
                            "post": {
                                "description": "Analyze an incident",
                                "operationId": "analyzeIncident",
                                "requestBody": {
                                    "required": True,
                                    "content": {
                                        "application/json": {
                                            "schema": {
                                                "type": "object",
                                                "properties": {
                                                    "incident_description": {
                                                        "type": "string",
                                                        "description": "Description of the incident"
                                                    },
                                                    "start_time": {
                                                        "type": "string",
                                                        "description": "Start time of the incident"
                                                    },
                                                    "end_time": {
                                                        "type": "string",
                                                        "description": "End time of the incident"
                                                    }
                                                },
                                                "required": ["incident_description"]
                                            }
                                        }
                                    }
                                },
                                "responses": {
                                    "200": {
                                        "description": "Analysis results",
                                        "content": {
                                            "application/json": {
                                                "schema": {
                                                    "type": "object",
                                                    "properties": {
                                                        "analysis": {
                                                            "type": "string"
                                                        },
                                                        "recommendations": {
                                                            "type": "array",
                                                            "items": {
                                                                "type": "string"
                                                            }
                                                        }
                                                    }
                                                }
                                            }
                                        }
                                    }
                                }
                            }
                        }
                    }
                }
            }
        ]
    }
    # ... Additional agents for CloudTrail, VPC, etc.
]

## 5. Knowledge Base Implementation

### 5.1 Serverless Knowledge Base with DynamoDB

**Location**: `/home/ec2-user/sre/sre_mcp/src/lambdas/knowledge-base-agent/lambda_function_serverless.py`

In [ ]:
class ServerlessKnowledgeBase:
    """Serverless knowledge base using DynamoDB instead of OpenSearch."""
    
    def __init__(self):
        self.kb_table = dynamodb.Table('sre-knowledge-base')
        self.vectors_table = dynamodb.Table('sre-knowledge-base-vectors')
        
    def generate_embedding(self, text: str) -> List[float]:
        """Generate embeddings using Amazon Titan."""
        try:
            response = bedrock_runtime.invoke_model(
                modelId='amazon.titan-embed-text-v1',
                contentType='application/json',
                accept='application/json',
                body=json.dumps({
                    "inputText": text[:8000]  # Truncate if too long
                })
            )
            
            result = json.loads(response['body'].read())
            return result['embedding']
            
        except Exception as e:
            logger.warning(f"Error generating embedding: {str(e)}")
            # Return mock embedding for demo
            return [random.random() for _ in range(1536)]
            
    def search_similar_documents(self, query: str, k: int = 5) -> List[Dict]:
        """Search for similar documents using vector similarity."""
        try:
            # Generate query embedding
            query_embedding = self.generate_embedding(query)
            
            # Get all documents
            response = self.kb_table.scan()
            documents = response.get('Items', [])
            
            # Calculate similarities
            similarities = []
            
            for doc in documents:
                # Get document embedding
                vector_response = self.vectors_table.get_item(
                    Key={'document_id': doc['document_id']}
                )
                
                if 'Item' in vector_response:
                    doc_embedding = vector_response['Item']['embedding']
                    
                    # Calculate cosine similarity
                    similarity = self.cosine_similarity(query_embedding, doc_embedding)
                    
                    similarities.append({
                        'document': doc,
                        'similarity': similarity
                    })
            
            # Sort by similarity and return top k
            similarities.sort(key=lambda x: x['similarity'], reverse=True)
            return [item['document'] for item in similarities[:k]]
            
        except Exception as e:
            logger.error(f"Error in search: {str(e)}")
            return []
    
    def cosine_similarity(self, vec1: List[float], vec2: List[float]) -> float:
        """Calculate cosine similarity between two vectors."""
        dot_product = sum(a * b for a, b in zip(vec1, vec2))
        norm1 = sum(a * a for a in vec1) ** 0.5
        norm2 = sum(b * b for b in vec2) ** 0.5
        
        if norm1 == 0 or norm2 == 0:
            return 0.0
            
        return float(dot_product / (norm1 * norm2))

### 5.2 Auto-Indexing OpsItems

When an OpsItem is created, it's automatically indexed to the knowledge base:

In [ ]:
# From lambda_function_serverless.py

def handle_opsitem_event(event, kb):
    """Handle OpsItem events from EventBridge."""
    try:
        # Extract OpsItem details from CloudTrail event
        detail = event['detail']
        request_params = detail.get('requestParameters', {})
        
        # Get OpsItem details
        ops_item_id = request_params.get('opsItemId')
        title = request_params.get('title', '')
        description = request_params.get('description', '')
        severity = request_params.get('severity', '3')
        
        # Create knowledge base document
        document = {
            'document_id': f'opsitem-{ops_item_id}',
            'title': title,
            'content': f"{title}\n\n{description}",
            'metadata': {
                'category': 'incident',
                'type': 'opsitem',
                'severity': severity,
                'tags': ['auto-indexed', 'incident', 'opsitem'],
                'source': 'systems-manager',
                'ops_item_id': ops_item_id
            }
        }
        
        # Index the document
        kb.index_document(document)
        logger.info(f"Successfully indexed OpsItem: {ops_item_id}")
        
        return {
            'statusCode': 200,
            'body': json.dumps({
                'message': f'Successfully indexed OpsItem {ops_item_id}',
                'document_id': document['document_id']
            })
        }
        
    except Exception as e:
        logger.error(f"Error handling OpsItem event: {str(e)}")
        return {
            'statusCode': 500,
            'body': json.dumps({'error': str(e)})
        }

## 6. Complete Demo Walkthrough

### Step-by-Step Demo Flow

### Step 3: Automatic AI Analysis and Knowledge Base Indexing

**NEW AUTOMATIC FLOW**:
1. EventBridge detects OpsItem creation via CloudTrail
2. Triggers enhanced `sre-opsitem-indexer` Lambda
3. OpsItem Indexer automatically:
   - Indexes incident to Knowledge Base
   - Invokes Supervisor Lambda for AI analysis
   - Supervisor automatically invokes ALL agent Lambdas
   - Generates comprehensive root cause analysis
   - Updates OpsItem with results
   - Stores analysis in Knowledge Base

**No user action required!** The entire AI analysis happens automatically within minutes of incident creation.

### Step 2: Generate an Incident

1. Navigate to "Incident Management" tab
2. Click "🎲 Generate Random Incident"
3. System will:
   - Create CloudWatch metrics (CPU spike, memory usage)
   - Generate CloudWatch logs with errors
   - Create OpsItem in Systems Manager
   - Display incident details

### Step 3: Automatic Knowledge Base Indexing

EventBridge automatically detects OpsItem creation:
- Triggers `sre-knowledge-base-agent-lambda`
- Indexes incident for future searches
- No user action required

### Step 4: Analyze Root Cause

1. Navigate to "Analyze Incident" tab
2. Select the incident from dropdown
3. Click "🔍 Analyze Root Cause"
4. System will:
   - Invoke supervisor Lambda
   - Gather CloudWatch metrics and logs
   - Search knowledge base for similar incidents
   - Call Bedrock AI for analysis
   - Invoke specialized agent Lambdas
   - Display comprehensive analysis

## Complete Automatic AI Analysis Flow

Here's the enhanced end-to-end flow showing automatic AI analysis:

```
User Creates Incident (Streamlit)
           │
           ▼
    OpsItem Created
           │
           ▼
    CloudTrail Event
           │
           ▼
┌──────────────────┐
│  EventBridge Rule │ (sre-opsitem-indexing)
└────────┬─────────┘
         │
         ▼
┌──────────────────────┐
│ OpsItem Indexer      │ (Enhanced)
│ Lambda               │
└────────┬─────────────┘
         │
         ├─────────────────────────────┐
         │                             │
         ▼                             ▼
┌─────────────────┐          ┌──────────────────┐
│ Knowledge Base  │          │ Supervisor       │
│ Lambda          │          │ Lambda           │
│ (Index OpsItem) │          │ (Auto Analysis)  │
└─────────────────┘          └────────┬─────────┘
                                      │
                    ┌─────────────────┼─────────────────┐
                    │                 │                 │
                    ▼                 ▼                 ▼
         ┌──────────────┐  ┌──────────────┐  ┌──────────────┐
         │ CloudWatch   │  │ CloudTrail   │  │ VPC/Flow     │
         │ Logs Agent   │  │ Agent        │  │ Logs Agents  │
         └──────────────┘  └──────────────┘  └──────────────┘
                    │                 │                 │
                    ▼                 ▼                 ▼
         ┌──────────────┐  ┌──────────────┐  ┌──────────────┐
         │ Trusted      │  │ Personal     │  │ Knowledge    │
         │ Advisor      │  │ Health Agent │  │ Base Search  │
         └──────────────┘  └──────────────┘  └──────────────┘
                    │                 │                 │
                    └─────────────────┴─────────────────┘
                                      │
                                      ▼
                            ┌──────────────────┐
                            │ Bedrock AI       │
                            │ (Claude 3)       │
                            │ Root Cause       │
                            │ Analysis         │
                            └────────┬─────────┘
                                     │
                                     ▼
                            ┌──────────────────┐
                            │ Update OpsItem   │
                            │ Store Analysis   │
                            │ in KB            │
                            └──────────────────┘
```

**Key Points**:
- Entire flow is AUTOMATIC - no manual intervention
- ALL agents are invoked for comprehensive analysis
- Results stored in both OpsItem and Knowledge Base
- Analysis available in Streamlit dashboard
- Complete audit trail maintained

### Step 5: View Analysis Results

The dashboard displays:
- **Root Cause Analysis**: AI-generated analysis from Bedrock
- **Impact Assessment**: Business and technical impact
- **Immediate Steps**: Actionable mitigation steps
- **Long-term Recommendations**: Prevention measures
- **Similar Incidents**: Historical context from KB
- **Metrics Visualization**: Charts showing anomalies
- **Log Analysis**: Error patterns and messages

## Key Points Summary

1. **No Grafana Integration**: The system uses CloudWatch directly, not Grafana
2. **No Automatic Alarms**: CloudWatch alarms are not created automatically
3. **Manual Trigger Required**: Root cause analysis requires user action
4. **EventBridge**: Only used for KB auto-indexing, not analysis triggers
5. **AI-Powered**: Uses AWS Bedrock Claude 3 Sonnet for intelligent analysis
6. **Serverless**: All components use Lambda and DynamoDB (no servers)
7. **Cost-Effective**: DynamoDB KB costs <$10/month vs $70+ for OpenSearch
8. **Comprehensive**: Analyzes metrics, logs, and historical incidents
9. **Modular**: Specialized agents for different AWS services
10. **Extensible**: Can add new agents and integrations easily